In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate

import numpy as np
import torch
import pandas as pd
import json
import shutil

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

In [ ]:
train_df = pd.read_csv('/kaggle/input/datasets/mohamed00mamdouh/aspect-dataset/aspect_train_df.csv')
val_df = pd.read_csv('/kaggle/input/datasets/mohamed00mamdouh/aspect-dataset/aspect_val_df.csv')

In [ ]:
model_name = "xlm-roberta-base"

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {v: k for k, v in label2id.items()}

train_data = train_df[["text_for_sentiment", "aspect", "label_for_sentiment"]].copy()
val_data   = val_df[["text_for_sentiment", "aspect", "label_for_sentiment"]].copy()

train_data["text_for_sentiment"] = train_data["text_for_sentiment"].astype(str)
val_data["text_for_sentiment"] = val_data["text_for_sentiment"].astype(str)

train_data["aspect"] = train_data["aspect"].astype(str)
val_data["aspect"] = val_data["aspect"].astype(str)

train_data["label"] = train_data["label_for_sentiment"].map(label2id)
val_data["label"] = val_data["label_for_sentiment"].map(label2id)

train_data = train_data.dropna(subset=["label"])
val_data = val_data.dropna(subset=["label"])

train_data["label"] = train_data["label"].astype(int)
val_data["label"] = val_data["label"].astype(int)

In [ ]:
train_data["input_text"] = (
    train_data["text_for_sentiment"] +
    " </s> aspect: " +
    train_data["aspect"]
)

val_data["input_text"] = (
    val_data["text_for_sentiment"] +
    " </s> aspect: " +
    val_data["aspect"]
)

In [ ]:
train_dataset = Dataset.from_pandas(train_data[["input_text", "label"]])
val_dataset = Dataset.from_pandas(val_data[["input_text", "label"]])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_sentiment_data(example):
    encoding = tokenizer(
        example["input_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    encoding["labels"] = example["label"]
    return encoding

train_dataset = train_dataset.map(tokenize_sentiment_data)
val_dataset = val_dataset.map(tokenize_sentiment_data)

cols_to_keep = ["input_ids", "attention_mask", "labels"]

train_dataset.set_format(type="torch", columns=cols_to_keep)
val_dataset.set_format(type="torch", columns=cols_to_keep)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./sentiment_model",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,
    num_train_epochs=50,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",

    fp16=torch.cuda.is_available(),

    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=5,
            early_stopping_threshold=0.0
        )
    ]
)

train_result = trainer.train()

In [ ]:
trainer.save_model("./best_sentiment_model")
tokenizer.save_pretrained("./best_sentiment_model")

with open("./best_sentiment_model/label2id.json", "w") as f:
    json.dump(label2id, f)

with open("./best_sentiment_model/id2label.json", "w") as f:
    json.dump(id2label, f)

shutil.make_archive(
    "/kaggle/working/best_sentiment_model",
    "zip",
    "./best_sentiment_model"
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import json

model_path = "./best_sentiment_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.eval()

def predict_sentiment(text, aspect):
    input_text = str(text) + " </s> aspect: " + str(aspect)

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()
    pred_id = int(np.argmax(probs))

    return {
        "label": model.config.id2label[pred_id],
        "confidence": float(probs[pred_id]),
        "probs": {
            model.config.id2label[i]: float(probs[i])
            for i in range(len(probs))
        }
    }

In [ ]:
predict_sentiment(
    "لا يوجد الدفع بالبطاقة عند الاستلام",
    "app_experience"
)